# Bloque 1: Configuración e Importación de dependencias

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import os
import csv
import json

In [ ]:
VIDEO_PATH = "VideosAnalisis\clip 1 ‐ Hecho con Clipchamp.mp4"
MAPA_PATH = "beachvolleyballcourt.png"
MODEL_PATH = "yolo11n.pt"
MARGIN_PERCENT = 0.10
EXPECTED_PLAYERS = 4

print("✓ Configuración cargada")

model = YOLO(MODEL_PATH)
print("✓ Modelo YOLO cargado")


## Definición de funciones auxiliares

In [ ]:
def get_points(event, x, y, flags, params):
    """Callback para seleccionar puntos con el mouse."""
    points = params["points"]
    image = params["image"]
    wname = params["wname"]
    max_points = params["max_points"]

    if event == cv2.EVENT_LBUTTONDOWN and len(points) < max_points:
        points.append([x, y])
        cv2.circle(image, (x, y), 6, (0, 0, 255), -1)
        cv2.putText(image, str(len(points)), (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        cv2.imshow(wname, image)

        if len(points) == max_points:
            cv2.waitKey(300)
            cv2.destroyWindow(wname)

def point_in_polygon_with_margin(point, polygon, margin_percent=0.10):
    """Verifica si un punto está dentro de un polígono expandido."""
    center = np.mean(polygon, axis=0)
    expanded_polygon = []
    max_y = np.max(polygon[:, 1])
    
    for pt in polygon:
        if abs(pt[1] - max_y) < 5:
            direction = pt - center
            direction[1] = min(0, direction[1])
            expanded_pt = pt + direction * margin_percent
        else:
            direction = pt - center
            expanded_pt = pt + direction * margin_percent
        expanded_polygon.append(expanded_pt)
    
    expanded_polygon = np.array(expanded_polygon, dtype=np.int32)
    result = cv2.pointPolygonTest(expanded_polygon, point, False)
    return result >= 0


def calculate_iou(box1, box2):
    """Calcula Intersection over Union entre dos bounding boxes."""
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    if x2_i < x1_i or y2_i < y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0

def get_track_info(tracking_data, track_id):
    """Obtiene información completa de un track."""
    frames = []
    positions = []
    bboxes = []
    
    for frame_idx, detections in tracking_data.items():
        for det in detections:
            if det[0] == track_id:
                frames.append(frame_idx)
                positions.append((det[5], det[6]))
                bboxes.append((det[1], det[2], det[3], det[4]))
    
    if not frames:
        return None, None, [], []
    
    return min(frames), max(frames), positions, bboxes

def should_merge_ids(tracking_data, id1, id2, max_gap=5, max_distance=150, min_iou=0.3):
    """Determina si dos IDs deberían fusionarse."""
    first1, last1, positions1, bboxes1 = get_track_info(tracking_data, id1)
    first2, last2, positions2, bboxes2 = get_track_info(tracking_data, id2)
    
    if first1 is None or first2 is None:
        return False
    
    # CASO 1: Secuencial
    if last1 < first2:
        gap = first2 - last1
        if gap <= max_gap:
            last_pos1 = positions1[-1]
            first_pos2 = positions2[0]
            distance = np.sqrt((last_pos1[0] - first_pos2[0])**2 + 
                              (last_pos1[1] - first_pos2[1])**2)
            
            last_bbox1 = bboxes1[-1]
            first_bbox2 = bboxes2[0]
            size1 = (last_bbox1[2] - last_bbox1[0]) * (last_bbox1[3] - last_bbox1[1])
            size2 = (first_bbox2[2] - first_bbox2[0]) * (first_bbox2[3] - first_bbox2[1])
            size_ratio = min(size1, size2) / max(size1, size2) if max(size1, size2) > 0 else 0
            
            if distance <= max_distance and size_ratio > 0.5:
                return True
    
    # CASO 2: Solapamiento
    overlap_start = max(first1, first2)
    overlap_end = min(last1, last2)
    
    if overlap_start <= overlap_end:
        overlapping_frames = []
        for frame_idx in range(overlap_start, overlap_end + 1):
            if frame_idx not in tracking_data:
                continue
            
            bbox1, bbox2 = None, None
            pos1, pos2 = None, None
            
            for det in tracking_data[frame_idx]:
                if det[0] == id1:
                    bbox1 = (det[1], det[2], det[3], det[4])
                    pos1 = (det[5], det[6])
                if det[0] == id2:
                    bbox2 = (det[1], det[2], det[3], det[4])
                    pos2 = (det[5], det[6])
            
            if bbox1 and bbox2:
                distance = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
                iou = calculate_iou(bbox1, bbox2)
                overlapping_frames.append((distance, iou))
        
        if overlapping_frames:
            avg_distance = np.mean([d for d, _ in overlapping_frames])
            avg_iou = np.mean([iou for _, iou in overlapping_frames])
            
            if avg_iou >= min_iou or avg_distance <= max_distance * 0.5:
                return True
    
    return False

def merge_track_ids(tracking_data, id_from, id_to):
    """Fusiona id_from en id_to."""
    for frame_idx in tracking_data:
        new_detections = []
        for det in tracking_data[frame_idx]:
            if det[0] == id_from:
                new_det = (id_to,) + det[1:]
                new_detections.append(new_det)
            else:
                new_detections.append(det)
        tracking_data[frame_idx] = new_detections

def count_frames_with_excess(tracking_data, max_expected=4):
    """Cuenta frames con más jugadores del esperado."""
    return sum(1 for dets in tracking_data.values() if len(dets) > max_expected)

print("✓ Funciones auxiliares definidas")


# Bloque 2: Obtención de POI

In [ ]:
video = cv2.VideoCapture(VIDEO_PATH)
if not video.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"✓ Video cargado: {width}x{height}, {fps:.1f} FPS, {total_frames} frames")

ret, first_frame = video.read()
if not ret:
    raise RuntimeError("No se pudo leer el primer frame")

mapa = cv2.imread(MAPA_PATH)
if mapa is None:
    raise FileNotFoundError(f"No se pudo cargar el mapa: {MAPA_PATH}")

print(f"✓ Mapa cargado: {mapa.shape[1]}x{mapa.shape[0]}")
puntos_campo = []
N = 4

imgA = first_frame.copy()
cv2.namedWindow("Selecciona 4 esquinas del campo", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona 4 esquinas del campo", 1200, 800)
cv2.imshow("Selecciona 4 esquinas del campo", imgA)

cv2.setMouseCallback(
    "Selecciona 4 esquinas del campo",
    get_points,
    {"points": puntos_campo, "image": imgA, 
     "wname": "Selecciona 4 esquinas del campo", "max_points": N}
)

print("Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)")
cv2.waitKey(0)
cv2.destroyAllWindows()

puntos_campo = np.array(puntos_campo, dtype=np.float32)
print(f"✓ {len(puntos_campo)} puntos seleccionados")

# --- Selección de puntos correspondientes en el MAPA ---
puntos_mapa = []

imgB = mapa.copy()
cv2.namedWindow("Selecciona las MISMAS 4 esquinas en el mapa", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona las MISMAS 4 esquinas en el mapa", 800, 600)
cv2.imshow("Selecciona las MISMAS 4 esquinas en el mapa", imgB)

cv2.setMouseCallback(
    "Selecciona las MISMAS 4 esquinas en el mapa",
    get_points,
    {"points": puntos_mapa, "image": imgB,
     "wname": "Selecciona las MISMAS 4 esquinas en el mapa", "max_points": 4}
)

cv2.waitKey(0)
cv2.destroyAllWindows()

puntos_mapa = np.array(puntos_mapa, dtype=np.float32)

assert len(puntos_campo) == 4 and len(puntos_mapa) == 4, "Error en selección de puntos"

H, status = cv2.findHomography(puntos_campo, puntos_mapa, cv2.RANSAC)

if H is None:
    raise RuntimeError("No se pudo calcular la homografía")

print("✓ Homografía calculada correctamente")


# Bloque 3: Detección de Jugadores

In [ ]:
video.set(cv2.CAP_PROP_POS_FRAMES, 0)
tracking_data = {}

print(f"\nProcesando {total_frames} frames...")
print("Esto puede tardar unos minutos...\n")

frame_idx = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    results = model.track(frame, persist=True, verbose=False, classes=[0])
    frame_detections = []
    
    for r in results:
        if r.boxes.id is None:
            continue
            
        for box, track_id in zip(r.boxes, r.boxes.id):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = (x1 + x2) // 2
            cy = y2
            
            if point_in_polygon_with_margin((cx, cy), puntos_campo, MARGIN_PERCENT):
                frame_detections.append((
                    int(track_id.item()),
                    x1, y1, x2, y2,
                    cx, cy
                ))
    
    tracking_data[frame_idx] = frame_detections
    
    if frame_idx % 50 == 0:
        progress = (frame_idx / total_frames) * 100
        print(f"  Frame {frame_idx}/{total_frames} ({progress:.1f}%) - {len(frame_detections)} jugadores detectados")
    
    frame_idx += 1

print(f"\n✓ Tracking completado: {len(tracking_data)} frames procesados")

# Estadísticas iniciales
all_track_ids = set()
for dets in tracking_data.values():
    for det in dets:
        all_track_ids.add(det[0])

total_detections = sum(len(dets) for dets in tracking_data.values())
print(f"  Total detecciones: {total_detections}")
print(f"  IDs únicos detectados: {len(all_track_ids)}")
print(f"  IDs: {sorted(all_track_ids)}")


## Corrección

Elimininación de duplicados

In [ ]:
print("\n" + "=" * 70)
print("CORRECCIÓN AVANZADA DE IDS DUPLICADOS")
print("=" * 70)

all_ids = sorted(all_track_ids)
print(f"\n📊 Estado inicial:")
print(f"   IDs detectados: {all_ids}")
print(f"   Total IDs: {len(all_ids)}")
print(f"   Frames con >4 jugadores: {count_frames_with_excess(tracking_data)}")

merge_map = {id: id for id in all_ids}
total_merges = 0

params = [
    (5, 100, 0.4, "Muy estricto - gaps pequeños"),
    (10, 150, 0.3, "Estricto - gaps medianos"),
    (15, 200, 0.25, "Moderado - gaps más largos"),
    (20, 250, 0.2, "Permisivo - oclusiones largas"),
    (30, 350, 0.15, "Muy permisivo - último intento"),
]

for iteration, (max_gap, max_distance, min_iou, description) in enumerate(params, 1):
    print(f"\n{'─' * 70}")
    print(f"ITERACIÓN {iteration}: {description}")
    print(f"   Parámetros: gap≤{max_gap}f, dist≤{max_distance}px, IoU≥{min_iou}")
    print(f"{'─' * 70}")
    
    current_ids = set()
    for dets in tracking_data.values():
        for det in dets:
            current_ids.add(det[0])
    current_ids = sorted(current_ids)
    
    frames_excess = count_frames_with_excess(tracking_data)
    
    print(f"   IDs actuales: {current_ids} ({len(current_ids)} IDs)")
    print(f"   Frames problemáticos: {frames_excess}")
    
    if len(current_ids) <= EXPECTED_PLAYERS and frames_excess < 10:
        print(f"   ✅ ¡Objetivo alcanzado!")
        break
    
    merge_candidates = []
    
    for i, id1 in enumerate(current_ids):
        for id2 in current_ids[i+1:]:
            if should_merge_ids(tracking_data, id1, id2, max_gap, max_distance, min_iou):
                impact = 0
                for frame_idx, dets in tracking_data.items():
                    ids_in_frame = [d[0] for d in dets]
                    if id1 in ids_in_frame and id2 in ids_in_frame:
                        impact += 1
                
                merge_candidates.append((id1, id2, impact))
    
    merge_candidates.sort(key=lambda x: x[2], reverse=True)
    print(f"   Candidatos encontrados: {len(merge_candidates)}")
    
    if not merge_candidates:
        print(f"   ⚠️  No se encontraron fusiones posibles")
        continue
    
    merges_in_iteration = 0
    for id1, id2, impact in merge_candidates:
        current_id1 = merge_map.get(id1, id1)
        current_id2 = merge_map.get(id2, id2)
        
        if current_id1 == current_id2:
            continue
        
        merge_from = max(current_id1, current_id2)
        merge_to = min(current_id1, current_id2)
        
        print(f"      → Fusionando ID {merge_from} → ID {merge_to} (resuelve {impact} frames)")
        
        merge_track_ids(tracking_data, merge_from, merge_to)
        
        for key in merge_map:
            if merge_map[key] == merge_from:
                merge_map[key] = merge_to
        merge_map[merge_from] = merge_to
        
        merges_in_iteration += 1
        total_merges += 1
    
    print(f"   ✓ Fusiones realizadas: {merges_in_iteration}")

print(f"\n{'=' * 70}")
print("✅ CORRECCIÓN COMPLETADA")
print(f"{'=' * 70}")

final_ids = set()
for dets in tracking_data.values():
    for det in dets:
        final_ids.add(det[0])
final_ids = sorted(final_ids)

print(f"\n📊 Resumen de cambios:")
print(f"   IDs originales: {len(all_ids)} → IDs finales: {len(final_ids)}")
print(f"   Total de fusiones: {total_merges}")
print(f"   IDs finales: {final_ids}")

print(f"\n📈 Distribución de jugadores por frame:")
distribution = defaultdict(int)
for dets in tracking_data.values():
    distribution[len(dets)] += 1

for num_players in sorted(distribution.keys()):
    count = distribution[num_players]
    percentage = (count / len(tracking_data)) * 100
    bar = "█" * int(percentage / 2)
    marker = "✓" if num_players == EXPECTED_PLAYERS else "⚠" if num_players > EXPECTED_PLAYERS else "!"
    print(f"   {marker} {num_players} jugadores: {count:4d} frames ({percentage:5.1f}%) {bar}")

frames_exact = distribution.get(EXPECTED_PLAYERS, 0)
frames_over = sum(count for num, count in distribution.items() if num > EXPECTED_PLAYERS)
frames_under = sum(count for num, count in distribution.items() if num < EXPECTED_PLAYERS)

print(f"\n🎯 Calidad del tracking:")
print(f"   Frames perfectos (4 jugadores): {frames_exact} ({frames_exact/len(tracking_data)*100:.1f}%)")
print(f"   Frames con exceso (>4): {frames_over} ({frames_over/len(tracking_data)*100:.1f}%)")
print(f"   Frames con déficit (<4): {frames_under} ({frames_under/len(tracking_data)*100:.1f}%)")

if len(final_ids) == EXPECTED_PLAYERS:
    print(f"\n🎉 ¡PERFECTO! Exactamente {EXPECTED_PLAYERS} jugadores detectados")
elif len(final_ids) < EXPECTED_PLAYERS:
    print(f"\n⚠️  Solo {len(final_ids)} jugadores detectados (esperados: {EXPECTED_PLAYERS})")
else:
    print(f"\n⚠️  {len(final_ids)} jugadores detectados (esperados: {EXPECTED_PLAYERS})")

all_track_ids = final_ids


In [ ]:
print(f"\n{'=' * 70}")
print("ANÁLISIS FINAL DEL TRACKING")
print(f"{'=' * 70}")

track_durations_final = defaultdict(int)

for frame_idx, detections in tracking_data.items():
    for det in detections:
        track_id = det[0]
        track_durations_final[track_id] += 1

print("\nDuración de cada track (en frames):")
for track_id in sorted(track_durations_final.keys()):
    duration = track_durations_final[track_id]
    duration_sec = duration / fps
    percentage = (duration / total_frames) * 100
    print(f"  ID {track_id}: {duration} frames ({duration_sec:.1f}s, {percentage:.1f}% del video)")

print(f"\n📊 Resumen:")
print(f"  • Jugadores únicos: {len(final_ids)}")
print(f"  • Frame más poblado: {max(len(dets) for dets in tracking_data.values())} jugadores")
print(f"  • Frame menos poblado: {min(len(dets) for dets in tracking_data.values())} jugadores")


# Bloque 4: Exportación de datos a CSV

In [ ]:
print(f"\n{'=' * 70}")
print("EXPORTANDO DATOS")
print(f"{'=' * 70}\n")

tracking_list = []

for frame_idx, detections in sorted(tracking_data.items()):
    for det in detections:
        track_id, x1, y1, x2, y2, cx, cy = det
        tracking_list.append({
            'frame': frame_idx,
            'track_id': track_id,
            'bbox_x1': x1,
            'bbox_y1': y1,
            'bbox_x2': x2,
            'bbox_y2': y2,
            'center_x': cx,
            'center_y': cy,
            'timestamp_sec': frame_idx / fps
        })

csv_filename = "tracking_data.csv"
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['frame', 'track_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 
                  'center_x', 'center_y', 'timestamp_sec']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(tracking_list)

print(f"✓ Datos exportados a '{csv_filename}'")
print(f"  Total de registros: {len(tracking_list)}")

# Actualización del JSON para incluir puntos del mapa
summary_data = {
    'video_info': {
        'fps': fps,
        'total_frames': total_frames,
        'width': width,
        'height': height,
        'video_path': VIDEO_PATH
    },
    'homografia': {
        'puntos_campo': puntos_campo.tolist(),
        'puntos_mapa': puntos_mapa.tolist(), # <--- Nuevo
        'matrix_H': H.tolist()               # <--- Matriz lista para reuso
    },
    'jugadores_ids': sorted(final_ids),
    'num_jugadores': len(final_ids)
}

json_filename = "tracking_summary.json"
with open(json_filename, 'w', encoding='utf-8') as jsonfile:
    json.dump(summary_data, jsonfile, indent=2)

print(f"✓ Resumen completo exportado a '{json_filename}'")

print(f"\n📁 Archivos generados en: {os.getcwd()}")


# Representación del resultado

In [ ]:
def visualizar_resultado_fluido(video_path, mapa_path, tracking_data, homography_matrix):
    cv2.destroyAllWindows()
    cap = cv2.VideoCapture(video_path)
    mapa_img = cv2.imread(mapa_path)
    
    if not cap.isOpened() or mapa_img is None:
        print("❌ Error: No se pudo cargar el video o el mapa.")
        return

    WINDOW_NAME = "Analisis Beach Volley - Vista Fluida"
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
    
    # Parámetros de estilo y escala
    TARGET_VIDEO_WIDTH = 1000 
    MINIMAP_WIDTH = 550
    COLOR_PASTEL = (240, 207, 137) # El azul glaciar que nos gustó
    
    # --- MEMORIA PARA FLUIDEZ ---
    # Guardará {track_id: (mx, my)}
    last_known_positions = {}

    print("▶️ Reproduciendo con persistencia de movimiento... [Q] para salir.")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        mapa_out = mapa_img.copy()
        ids_en_este_frame = []

        # 1. Procesar detecciones actuales
        if frame_idx in tracking_data:
            for det in tracking_data[frame_idx]:
                tid, x1, y1, x2, y2, cx, cy = det
                ids_en_este_frame.append(tid)
                
                # Dibujo en Video
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(frame, (cx, cy), 6, (0, 255, 0), -1)
                cv2.putText(frame, f"ID {tid}", (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                # Proyección y actualización de memoria
                pt = np.array([[cx, cy]], dtype=np.float32).reshape(-1, 1, 2)
                punto_proyectado = cv2.perspectiveTransform(pt, homography_matrix).reshape(-1, 2)[0]
                mx, my = int(punto_proyectado[0]), int(punto_proyectado[1])
                last_known_positions[tid] = (mx, my)

        # 2. Dibujar en el Mapa (Actuales + Persistentes)
        for tid, (mx, my) in last_known_positions.items():
            # Si el ID no está en este frame, lo dibujamos un poco más pequeño/transparente
            es_fantasma = tid not in ids_en_este_frame
            radio = 25
            color = COLOR_PASTEL if not es_fantasma else (200, 200, 200) # Gris si se perdió
            
            # Dibujo estético del punto
            cv2.circle(mapa_out, (mx, my), radio + 4, (255, 255, 255), -1) # Borde blanco
            cv2.circle(mapa_out, (mx, my), radio, color, -1)
            cv2.putText(mapa_out, str(tid), (mx - 10, my + 8), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2, cv2.LINE_AA)

        # 3. Composición y Redimensionado
        h_v, w_v = frame.shape[:2]
        scale_v = TARGET_VIDEO_WIDTH / w_v
        frame_res = cv2.resize(frame, (TARGET_VIDEO_WIDTH, int(h_v * scale_v)))
        
        h_m, w_m = mapa_out.shape[:2]
        scale_m = MINIMAP_WIDTH / w_m
        mapa_res = cv2.resize(mapa_out, (MINIMAP_WIDTH, int(h_m * scale_m)))
        
        height_final = max(frame_res.shape[0], mapa_res.shape[0])
        
        def add_padding(img, target_h):
            h, w = img.shape[:2]
            return cv2.copyMakeBorder(img, (target_h-h)//2, target_h-h-(target_h-h)//2, 0, 0, 
                                     cv2.BORDER_CONSTANT, value=(20, 20, 20))

        combined_view = cv2.hconcat([add_padding(frame_res, height_final), 
                                     add_padding(mapa_res, height_final)])
        
        cv2.putText(combined_view, f"Frame: {frame_idx}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        cv2.imshow(WINDOW_NAME, combined_view)
        if cv2.waitKey(25) & 0xFF in [27, ord('q')]: break
        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Visualización fluida finalizada.")

# Ejecutar la celda
visualizar_resultado_fluido(VIDEO_PATH, MAPA_PATH, tracking_data, H)

In [1]:
# ============================================================
# DEBUG PASO A PASO: DETECCIÓN DEL CAMPO POR ARENA
# ============================================================

import cv2
import numpy as np

VIDEO_PATH = r"VideosAnalisis\clip 9 ‐ Hecho con Clipchamp.mp4"
NUM_FRAMES = 180

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "No se pudo abrir el vídeo"

# ---------- 1. Imagen media ----------
acc = None
count = 0

while count < NUM_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    frame_f = frame.astype(np.float32)
    acc = frame_f if acc is None else acc + frame_f
    count += 1

cap.release()
avg = (acc / count).astype(np.uint8)

cv2.imshow("1 - Imagen media", avg)
cv2.waitKey(0)

# ---------- 2. Conversión a HSV ----------
hsv = cv2.cvtColor(avg, cv2.COLOR_BGR2HSV)

# ---------- 3. Segmentación de ARENA ----------
# Arena: tonos amarillos claros, baja saturación media
lower_sand = np.array([10, 20, 160])
upper_sand = np.array([40, 140, 255])

mask_sand = cv2.inRange(hsv, lower_sand, upper_sand)

cv2.imshow("3 - Máscara arena (bruta)", mask_sand)
cv2.waitKey(0)


# ---------- 5. Contornos ----------
contours, _ = cv2.findContours(mask_sand, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

dbg_contours = avg.copy()
for c in contours:
    cv2.drawContours(dbg_contours, [c], -1, (0, 255, 255), 2)

cv2.imshow("5 - Contornos arena", dbg_contours)
cv2.waitKey(0)

# ---------- 6. Selección de la región de campo ----------
h, w = mask_sand.shape
candidatos = []

dbg_filtrado = avg.copy()

for c in contours:
    area = cv2.contourArea(c)
    if area < h * w * 0.1:
        continue

    x, y, cw, ch = cv2.boundingRect(c)
    cy = y + ch / 2
    aspect = cw / ch if ch > 0 else 0

    color = (0, 0, 255)  # rojo = descartado

    if cy > h * 0.55 and 1.3 < aspect < 3.5:
        candidatos.append(c)
        color = (0, 255, 0)

    cv2.rectangle(dbg_filtrado, (x, y), (x + cw, y + ch), color, 3)

cv2.imshow("6 - Filtrado geométrico arena", dbg_filtrado)
cv2.waitKey(0)

if not candidatos:
    raise RuntimeError("No se detectó la región de arena del campo")

campo = max(candidatos, key=cv2.contourArea)

# ---------- 7. Región de campo final ----------
dbg_campo = avg.copy()
cv2.drawContours(dbg_campo, [campo], -1, (0, 255, 0), 4)

cv2.imshow("7 - Región ARENA seleccionada", dbg_campo)
cv2.waitKey(0)

# ---------- 8. Rectángulo mínimo del campo ----------
rect = cv2.minAreaRect(campo)
box = cv2.boxPoints(rect)
box = box.astype(int)

dbg_rect = avg.copy()
cv2.drawContours(dbg_rect, [box], 0, (255, 0, 0), 4)

cv2.imshow("8 - Campo detectado (minAreaRect)", dbg_rect)
cv2.waitKey(0)

cv2.destroyAllWindows()
